In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import math

ГЛОБАЛЬНЫЕ ПАРАМЕТРЫ (Задание 1)

In [3]:
K_NEIGHBORS = 20  # Число k соседей для алгоритма
TOP_X = 10  # Сколько рекомендаций выводить
TEST_SIZE = 0.2  # Процент данных для контроля (20%)
MIN_RATINGS_PER_MOVIE = 50  # Фильтрация малоизвестных фильмов для стабильности

ЗАГРУЗКА ДАННЫХ

In [4]:
# Загрузка рейтингов
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('ratings.dat', sep='::', names=ratings_cols, engine='python', encoding='latin-1')

# Загрузка названий фильмов
movies_cols = ['movie_id', 'title', 'genres']
movies = pd.read_csv('movies.dat', sep='::', names=movies_cols, engine='python', encoding='latin-1')

ПОДГОТОВКА И ОБУЧЕНИЕ

In [5]:
print(f"Обучение модели (K={K_NEIGHBORS}, Test={TEST_SIZE * 100}%)...")

# Фильтруем фильмы, чтобы оставить только те, у которых достаточно оценок
movie_counts = ratings.groupby('movie_id').size()
popular_movies = movie_counts[movie_counts >= MIN_RATINGS_PER_MOVIE].index
filtered_ratings = ratings[ratings['movie_id'].isin(popular_movies)]

# Разделение на train и test (по рейтингам)
train_df, test_df = train_test_split(filtered_ratings, test_size=TEST_SIZE, random_state=42)

# Создаем сводную таблицу (Pivot Table) для Item-based
# Строки - фильмы, столбцы - пользователи
user_item_matrix = train_df.pivot(index='movie_id', columns='user_id', values='rating').fillna(0)

# Превращаем в разреженную матрицу для ускорения вычислений
sparse_matrix = csr_matrix(user_item_matrix.values)

# Обучаем модель KNN
# Используем косинусное сходство (cosine similarity) - стандарт для РС
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=K_NEIGHBORS, n_jobs=-1)
model_knn.fit(sparse_matrix)

Обучение модели (K=20, Test=20.0%)...


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",20
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1


ФУНКЦИЯ РЕКОМЕНДАЦИИ

In [6]:
def get_recommendations(movie_title, movies_df, model_knn, user_item_matrix):
    # Поиск ID фильма по названию
    try:
        movie_idx = movies_df[movies_df['title'].str.contains(movie_title, case=False, regex=False)].iloc[0]['movie_id']
    except IndexError:
        return "Фильм не найден в базе данных."

    if movie_idx not in user_item_matrix.index:
        return "Фильм был отфильтрован из-за малого количества оценок."

    # Получаем индекс строки в матрице
    query_index = user_item_matrix.index.get_loc(movie_idx)

    # Находим ближайших соседей
    distances, indices = model_knn.kneighbors(
        user_item_matrix.iloc[query_index, :].values.reshape(1, -1),
        n_neighbors=TOP_X + 1
    )

    print(f"\nРекомендации для фильма: {movie_title}")
    recs = []
    for i in range(1, len(distances.flatten())):
        target_movie_id = user_item_matrix.index[indices.flatten()[i]]
        title = movies_df[movies_df['movie_id'] == target_movie_id]['title'].values[0]
        dist = distances.flatten()[i]
        recs.append((title, dist))
        print(f"{i}: {title} (дистанция: {dist:.3f})")

    return recs

ОЦЕНКА КАЧЕСТВА (МЕТРИКИ)

In [7]:
"""
Расчет RMSE для предсказания рейтингов.
Для Item-based KNN: рейтинг = взвешенное среднее оценок соседей.
"""
print("\nРасчет метрик качества...")
y_true = []
y_pred = []

# Берем случайную выборку из теста для ускорения оценки
test_sample = test_df.sample(min(2000, len(test_df)), random_state=42)

for _, row in test_sample.iterrows():
    uid = row['user_id']
    mid = row['movie_id']

    if mid in user_item_matrix.index:
        # Находим соседей фильма mid
        query_index = user_item_matrix.index.get_loc(mid)
        distances, indices = model_knn.kneighbors(
            user_item_matrix.iloc[query_index, :].values.reshape(1, -1),
            n_neighbors=K_NEIGHBORS
        )

        neighbor_ids = user_item_matrix.index[indices.flatten()[1:]]
        neighbor_distances = distances.flatten()[1:]

        # Пытаемся предсказать рейтинг как среднее соседей, которых этот пользователь уже оценивал
        weights = []
        ratings = []

        for i, n_id in enumerate(neighbor_ids):
            # Если у пользователя в обучающей выборке есть оценка соседа
            user_rating = user_item_matrix.loc[n_id, uid] if uid in user_item_matrix.columns else 0
            if user_rating > 0:
                # Вес = 1 - дистанция (чем меньше дистанция, тем больше сходство)
                sim = 1 - neighbor_distances[i]
                weights.append(sim)
                ratings.append(user_rating)

        if sum(weights) > 0:
            predicted_rating = sum(np.array(ratings) * np.array(weights)) / sum(weights)
            y_true.append(row['rating'])
            y_pred.append(predicted_rating)

rmse = math.sqrt(mean_squared_error(y_true, y_pred))
print(f"Метрика RMSE: {rmse:.4f} (на базе {len(y_true)} предсказаний)")
print("--------------------------------------------------")


Расчет метрик качества...
Метрика RMSE: 0.9637 (на базе 1920 предсказаний)
--------------------------------------------------


Демонстрация

In [9]:
example_movie = "Toy Story (1995)"
get_recommendations(example_movie, movies, model_knn, user_item_matrix)


Рекомендации для фильма: Toy Story (1995)
1: Toy Story 2 (1999) (дистанция: 0.499)
2: Aladdin (1992) (дистанция: 0.519)
3: Groundhog Day (1993) (дистанция: 0.522)
4: Bug's Life, A (1998) (дистанция: 0.541)
5: Back to the Future (1985) (дистанция: 0.551)
6: Babe (1995) (дистанция: 0.558)
7: Star Wars: Episode V - The Empire Strikes Back (1980) (дистанция: 0.559)
8: Lion King, The (1994) (дистанция: 0.563)
9: Matrix, The (1999) (дистанция: 0.563)
10: Men in Black (1997) (дистанция: 0.564)


[('Toy Story 2 (1999)', np.float64(0.49864208941725263)),
 ('Aladdin (1992)', np.float64(0.5191210614342041)),
 ('Groundhog Day (1993)', np.float64(0.5220194086878087)),
 ("Bug's Life, A (1998)", np.float64(0.540860974240886)),
 ('Back to the Future (1985)', np.float64(0.5512076251483949)),
 ('Babe (1995)', np.float64(0.558107902250545)),
 ('Star Wars: Episode V - The Empire Strikes Back (1980)',
  np.float64(0.559218030918891)),
 ('Lion King, The (1994)', np.float64(0.5625795348225611)),
 ('Matrix, The (1999)', np.float64(0.5629573257001268)),
 ('Men in Black (1997)', np.float64(0.5644480734625699))]

In [14]:
example_movie_2 = "Indiana Jones and the Last Crusade"
get_recommendations(example_movie_2, movies, model_knn, user_item_matrix)


Рекомендации для фильма: Indiana Jones and the Last Crusade
1: Raiders of the Lost Ark (1981) (дистанция: 0.408)
2: Indiana Jones and the Temple of Doom (1984) (дистанция: 0.447)
3: Batman (1989) (дистанция: 0.454)
4: Die Hard (1988) (дистанция: 0.477)
5: Star Wars: Episode V - The Empire Strikes Back (1980) (дистанция: 0.478)
6: Star Wars: Episode VI - Return of the Jedi (1983) (дистанция: 0.483)
7: Terminator, The (1984) (дистанция: 0.493)
8: Star Wars: Episode IV - A New Hope (1977) (дистанция: 0.497)
9: Princess Bride, The (1987) (дистанция: 0.518)
10: Romancing the Stone (1984) (дистанция: 0.522)


[('Raiders of the Lost Ark (1981)', np.float64(0.40775401215260065)),
 ('Indiana Jones and the Temple of Doom (1984)',
  np.float64(0.4468354801132922)),
 ('Batman (1989)', np.float64(0.45384510330253836)),
 ('Die Hard (1988)', np.float64(0.4772398974753713)),
 ('Star Wars: Episode V - The Empire Strikes Back (1980)',
  np.float64(0.4783635054451225)),
 ('Star Wars: Episode VI - Return of the Jedi (1983)',
  np.float64(0.4831665349615416)),
 ('Terminator, The (1984)', np.float64(0.49274318613379453)),
 ('Star Wars: Episode IV - A New Hope (1977)',
  np.float64(0.49650113667745144)),
 ('Princess Bride, The (1987)', np.float64(0.517881979607014)),
 ('Romancing the Stone (1984)', np.float64(0.5215256794170043))]